# Preparação dos dados de treino — EUA

As features de data são calculadas a partir das datas de pesquisa e do voo, e os aeroportos e a companhia aérea são
codificados com target encoding. O encoder é ajustado só no conjunto de treino, para não passar informação do teste
para o modelo.

In [1]:
from pathlib import Path

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder

from flight_prices.itineraries import CATEGORICAL_FEATURES, DATE_FEATURES, TARGET, add_date_features, load_itineraries

DATA_DIR = Path("../data")

In [2]:
df = add_date_features(load_itineraries(DATA_DIR / "voos_filtrados.csv"))

X = df[CATEGORICAL_FEATURES + DATE_FEATURES]
y = df[TARGET].to_numpy()
del df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
preprocessor = make_pipeline(
    ColumnTransformer(
        [("target", TargetEncoder(target_type="continuous"), CATEGORICAL_FEATURES)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ),
    StandardScaler(),
)

X_train = preprocessor.fit_transform(X_train, y_train).astype(np.float32)
X_test = preprocessor.transform(X_test).astype(np.float32)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (17653510, 8)
Teste: (4413378, 8)


In [4]:
np.savez(
    DATA_DIR / "treino_eua.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    features=preprocessor.get_feature_names_out(),
)